# 음운 조건 검색

**작성일**: 2026-02-10  
**목적**: v7 dictionary에서 특정 음운 조건의 단어 검색

---

## 📚 목차
1. 환경 설정
2. v7 Dictionary 로드
3. 한글 자모 분석 함수
4. 종성 검색
5. 음운 환경 검색
6. 결과 저장

## 1️⃣ 환경 설정

In [16]:
# 1.1 Google Drive 마운트 (Colab에서만)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False
    print("로컬 환경에서 실행 중")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# 1.2 경로 설정
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
else:
    PROJECT_ROOT = 'g:/내 드라이브/DATA_2026'

V7_LEXICON = f'{PROJECT_ROOT}/10_dictionary_build/output/04_v7_lexicon.csv'
WORK_DIR = f'{PROJECT_ROOT}/30_search_dictionary'
RESULT_DIR = f'{WORK_DIR}/search_results'

print(f"✅ v7 Lexicon: {V7_LEXICON}")
print(f"✅ 결과 저장 폴더: {RESULT_DIR}")

✅ v7 Lexicon: /content/drive/MyDrive/DATA_2026/10_dictionary_build/output/04_v7_lexicon.csv
✅ 결과 저장 폴더: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results


In [18]:
# 1.3 라이브러리 임포트
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# 결과 폴더 생성
Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


## 2️⃣ v7 Dictionary 로드

In [19]:
# 2.1 전체 로드
df = pd.read_csv(V7_LEXICON, encoding='utf-8-sig')

print(f"✅ v7 Lexicon 로드 완료")
print(f"전체 행 수: {len(df):,}개")
print(f"컬럼 수: {len(df.columns)}개")

/tmp/ipython-input-2525405462.py:2: DtypeWarning: Columns (62) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(V7_LEXICON, encoding='utf-8-sig')


✅ v7 Lexicon 로드 완료
전체 행 수: 528,088개
컬럼 수: 66개


In [20]:
# 2.2 필수 컬럼 확인
essential_cols = ['sense_id', 'word', 'word_stem', 'pos', 'pos_tag', 'definition', 'freq_LS_total']

print("\n필수 컬럼 존재 여부:")
for col in essential_cols:
    exists = col in df.columns
    print(f"  {'✅' if exists else '❌'} {col}")


필수 컬럼 존재 여부:
  ✅ sense_id
  ✅ word
  ✅ word_stem
  ✅ pos
  ✅ pos_tag
  ✅ definition
  ✅ freq_LS_total


In [21]:
# 2.3 샘플 확인
df[essential_cols].head(10)

,sense_id,word,word_stem,pos,pos_tag,definition,freq_LS_total
0,45697:001,겅둥겅둥하다,겅둥겅둥하,동사,VV,긴 다리로 거볍게 계속해서 뛰다.,0
1,45697:002,겅둥겅둥하다,겅둥겅둥하,동사,VV,침착하지 못하고 거볍게 채신없이 행동하다.,0
2,45698:001,겅둥대다,겅둥대,동사,VV,긴 다리로 계속해서 거볍게 뛰다.,0
3,45698:002,겅둥대다,겅둥대,동사,VV,침착하지 못하고 채신없이 거볍게 행동하다.,0
4,45699:001,겅둥하다,겅둥하,형용사,VA,"입은 옷이, 아랫도리나 속옷이 드러날 정도로 매우 짧다.",0
5,45708:001,겅성드뭇,겅성드뭇,부사,MAG,많은 수효가 듬성듬성 흩어져 있는 모양.,0
6,45709:001,겅성드뭇이,겅성드뭇이,부사,MAG,많은 수효가 듬성듬성 흩어져 있는 상태로.,0
7,45710:001,겅성드뭇하다,겅성드뭇하,형용사,VA,많은 수효가 듬성듬성 흩어져 있다.,0
8,45714:001,겅정,겅정,부사,MAG,긴 다리를 모으고 거볍게 내뛰는 모양.,0
9,45715:001,겅정거리다,겅정거리,동사,VV,긴 다리를 모으고 거볍게 자꾸 내뛰다.,0


## 3️⃣ 한글 자모 분석 함수

In [22]:
# 3.1 종성 분석 함수
def get_final_consonant(char):
    """
    한글 음절의 종성 추출

    Parameters:
    - char (str): 한글 음절 하나 (예: '말')

    Returns:
    - int: 종성 인덱스 (0=없음, 1=ㄱ, 2=ㄲ, ..., 27=ㅎ)
    """
    if not char or len(char) != 1:
        return -1

    code = ord(char) - 0xAC00
    if code < 0 or code > 11171:  # 한글이 아님
        return -1

    return code % 28


def get_final_consonant_name(char):
    """
    종성 이름 반환

    Returns:
    - str: 종성 이름 (예: 'ㄹ', 'ㄴ', '없음')
    """
    finals = [
        '없음', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ',
        'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ',
        'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ',
        'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
    ]

    idx = get_final_consonant(char)
    if idx < 0 or idx >= len(finals):
        return '알수없음'
    return finals[idx]


# 테스트
test_chars = ['말', '물', '가', '집', '신']
print("종성 분석 테스트:")
for char in test_chars:
    final = get_final_consonant_name(char)
    print(f"  {char}: 종성 '{final}'")

종성 분석 테스트:
  말: 종성 'ㄹ'
  물: 종성 'ㄹ'
  가: 종성 '없음'
  집: 종성 'ㅂ'
  신: 종성 'ㄴ'


In [23]:
# 3.2 초성 분석 함수
def get_initial_consonant(char):
    """
    한글 음절의 초성 추출

    Returns:
    - int: 초성 인덱스 (0=ㄱ, 1=ㄲ, ..., 18=ㅎ)
    """
    if not char or len(char) != 1:
        return -1

    code = ord(char) - 0xAC00
    if code < 0 or code > 11171:
        return -1

    return code // (21 * 28)


def get_initial_consonant_name(char):
    """
    초성 이름 반환
    """
    initials = [
        'ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ',
        'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
    ]

    idx = get_initial_consonant(char)
    if idx < 0 or idx >= len(initials):
        return '알수없음'
    return initials[idx]


# 테스트
print("\n초성 분석 테스트:")
for char in test_chars:
    initial = get_initial_consonant_name(char)
    print(f"  {char}: 초성 '{initial}'")


초성 분석 테스트:
  말: 초성 'ㅁ'
  물: 초성 'ㅁ'
  가: 초성 'ㄱ'
  집: 초성 'ㅈ'
  신: 초성 'ㅅ'


## 4️⃣ 종성 검색

In [24]:
# 4.1 종성 검색 함수
def search_by_final_consonant(df, target_final, use_stem=True):
    """
    특정 종성으로 끝나는 단어 검색

    Parameters:
    - df: v7 lexicon DataFrame
    - target_final (str): 검색할 종성 (예: 'ㄹ', 'ㄴ')
    - use_stem (bool): word_stem 기준 검색 (기본값: True)

    Returns:
    - DataFrame: 검색 결과
    """
    column = 'word_stem' if use_stem else 'word'

    def matches_final(word):
        if not word:
            return False
        last_char = word[-1]
        return get_final_consonant_name(last_char) == target_final

    mask = df[column].apply(matches_final)
    return df[mask].copy()


# 4.2 ㄹ 종성 검색
result_rl = search_by_final_consonant(df, 'ㄹ')

print(f"\n✅ ㄹ 종성 단어: {len(result_rl):,}개")
print(f"\n상위 20개 (빈도순):")
result_rl_sorted = result_rl.sort_values('freq_LS_total', ascending=False)
result_rl_sorted[['word', 'word_stem', 'pos', 'freq_LS_total']].head(20)


✅ ㄹ 종성 단어: 33,440개

상위 20개 (빈도순):


,word,word_stem,pos,freq_LS_total
477086,일,일,의존 명사,15630
233068,말,말,명사,10750
233270,말다,말,동사,10750
455547,월,월,의존 명사,9092
477633,일다,일,동사,3862
477067,일,일,명사,3862
403894,알,알,명사,3266
404025,알다,알,동사,3266
469454,이날,이날,명사,3130
231902,만들다,만들,동사,2900


In [25]:
# 4.3 ㄴ 종성 검색
result_n = search_by_final_consonant(df, 'ㄴ')

print(f"\n✅ ㄴ 종성 단어: {len(result_n):,}개")
print(f"\n상위 20개 (빈도순):")
result_n_sorted = result_n.sort_values('freq_LS_total', ascending=False)
result_n_sorted[['word', 'word_stem', 'pos', 'freq_LS_total']].head(20)


✅ ㄴ 종성 단어: 58,543개

상위 20개 (빈도순):


,word,word_stem,pos,freq_LS_total
181178,년,년,의존 명사,19353
453632,원,원,의존 명사,9287
222536,때문,때문,의존 명사,6597
470665,이번,이번,명사,4333
468386,의원,의원,명사,4293
314400,북한,북한,명사,3618
499888,전,전,명사,3514
274920,번,번,의존 명사,3439
21364,관련,관련,명사,3199
147199,국민,국민,명사,2754


In [26]:
# 4.4 받침 없는 단어 검색
result_no_final = search_by_final_consonant(df, '없음')

print(f"\n✅ 받침 없는 단어: {len(result_no_final):,}개")
print(f"\n상위 20개 (빈도순):")
result_no_final_sorted = result_no_final.sort_values('freq_LS_total', ascending=False)
result_no_final_sorted[['word', 'word_stem', 'pos', 'freq_LS_total']].head(20)


✅ 받침 없는 단어: 277,974개

상위 20개 (빈도순):


,word,word_stem,pos,freq_LS_total
468902,이,이,명사,67730
298662,거,거,의존 명사,37313
103229,하다,하,동사,34092
369963,수,수,의존 명사,18126
496733,저,저,대명사,11408
522707,주,주,명사,10974
523189,주다,주,보조 동사,10974
468904,이,이,명사,10964
170273,나,나,대명사,10737
170542,나다,나,동사,10737


## 5️⃣ 음운 환경 검색

In [27]:
# 5.1 경음화 환경 검색
def search_tensification_environment(df):
    """
    경음화 환경: ㄱ,ㄷ,ㅂ 종성 + ㄱ,ㄷ,ㅂ,ㅅ,ㅈ 초성으로 시작하는 형태소

    seg_morph 필드를 활용 (형태소 분리 정보)
    """
    # seg_morph가 있는 경우만
    if 'seg_morph' not in df.columns:
        print("⚠️ seg_morph 컬럼이 없습니다.")
        return pd.DataFrame()

    # ㄱ,ㄷ,ㅂ 종성으로 끝나는 단어
    obstruent_finals = ['ㄱ', 'ㄷ', 'ㅂ']

    def has_tensification_env(row):
        word_stem = row.get('word_stem', '')
        if not word_stem:
            return False

        last_char = word_stem[-1]
        final = get_final_consonant_name(last_char)

        return final in obstruent_finals

    mask = df.apply(has_tensification_env, axis=1)
    return df[mask].copy()


result_tensif = search_tensification_environment(df)

print(f"\n✅ 경음화 환경 단어: {len(result_tensif):,}개")
print(f"\n상위 20개 (빈도순):")
result_tensif_sorted = result_tensif.sort_values('freq_LS_total', ascending=False)
result_tensif_sorted[['word', 'word_stem', 'pos', 'freq_LS_total']].head(20)


✅ 경음화 환경 단어: 62,069개

상위 20개 (빈도순):


,word,word_stem,pos,freq_LS_total
238162,먹다,먹,동사,11121
238116,먹,먹,명사,11121
348179,생각,생각,명사,5349
264929,받다,받,동사,4833
33692,집다,집,동사,3401
33543,집,집,명사,3401
385038,시작,시작,명사,3045
334321,사업,사업,명사,2940
161887,기업,기업,명사,2899
142066,교육,교육,명사,2862


In [28]:
# 5.2 비음화 환경 검색
def search_nasalization_environment(df):
    """
    비음화 환경: ㄱ,ㄷ,ㅂ 종성 (비음 아님)
    """
    # 경음화 환경과 동일 (실제로는 후행 형태소도 확인해야 하지만 간단히)
    return search_tensification_environment(df)


print("\n💡 비음화 환경은 경음화 환경과 유사 (후행 형태소에 따라 결정)")
print("   예: '국' + '물' → [궁물], '학' + '년' → [항년]")


💡 비음화 환경은 경음화 환경과 유사 (후행 형태소에 따라 결정)
   예: '국' + '물' → [궁물], '학' + '년' → [항년]


## 6️⃣ 결과 저장

In [29]:
# 6.1 타임스탬프 생성
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 6.2 ㄹ 종성 결과 저장
output_rl = f'{RESULT_DIR}/phonology_ㄹ종성_{timestamp}.csv'
result_rl_sorted.to_csv(output_rl, index=False, encoding='utf-8-sig')
print(f"✅ ㄹ 종성 결과 저장: {output_rl}")
print(f"   총 {len(result_rl_sorted):,}개")

# 6.3 ㄴ 종성 결과 저장
output_n = f'{RESULT_DIR}/phonology_ㄴ종성_{timestamp}.csv'
result_n_sorted.to_csv(output_n, index=False, encoding='utf-8-sig')
print(f"\n✅ ㄴ 종성 결과 저장: {output_n}")
print(f"   총 {len(result_n_sorted):,}개")

# 6.4 경음화 환경 결과 저장
output_tensif = f'{RESULT_DIR}/phonology_경음화환경_{timestamp}.csv'
result_tensif_sorted.to_csv(output_tensif, index=False, encoding='utf-8-sig')
print(f"\n✅ 경음화 환경 결과 저장: {output_tensif}")
print(f"   총 {len(result_tensif_sorted):,}개")

✅ ㄹ 종성 결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/phonology_ㄹ종성_20260210_081502.csv
   총 33,440개

✅ ㄴ 종성 결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/phonology_ㄴ종성_20260210_081502.csv
   총 58,543개

✅ 경음화 환경 결과 저장: /content/drive/MyDrive/DATA_2026/30_search_dictionary/search_results/phonology_경음화환경_20260210_081502.csv
   총 62,069개


In [30]:
# 6.5 요약 통계
summary = {
    'timestamp': timestamp,
    'total_words': len(df),
    'rl_final': len(result_rl),
    'n_final': len(result_n),
    'no_final': len(result_no_final),
    'tensification_env': len(result_tensif),
}

print("\n" + "="*60)
print("📊 검색 요약")
print("="*60)
print(f"전체 단어: {summary['total_words']:,}개")
print(f"ㄹ 종성: {summary['rl_final']:,}개 ({summary['rl_final']/summary['total_words']*100:.1f}%)")
print(f"ㄴ 종성: {summary['n_final']:,}개 ({summary['n_final']/summary['total_words']*100:.1f}%)")
print(f"받침 없음: {summary['no_final']:,}개 ({summary['no_final']/summary['total_words']*100:.1f}%)")
print(f"경음화 환경: {summary['tensification_env']:,}개 ({summary['tensification_env']/summary['total_words']*100:.1f}%)")
print("="*60)


📊 검색 요약
전체 단어: 528,088개
ㄹ 종성: 33,440개 (6.3%)
ㄴ 종성: 58,543개 (11.1%)
받침 없음: 277,974개 (52.6%)
경음화 환경: 62,069개 (11.8%)


---

## 📝 다음 단계

1. ✅ 음운 조건 검색 완료
2. ⏳ 형태소 조건 검색: `search_by_morpheme.ipynb`
3. ⏳ Seoul Corpus에서 실제 발음 확인

**결과 파일**: `search_results/phonology_*.csv`

In [31]:
# 생성된 파일 확인
import os
result_dir = f'{PROJECT_ROOT}/30_search_dictionary/search_results'
files = os.listdir(result_dir)
n_insertion_files = [f for f in files if 'n_insertion' in f]

print(f"search_results 폴더 파일 수: {len(files)}개\n")
print("n_insertion 관련 파일:")
for f in sorted(n_insertion_files):
    full_path = os.path.join(result_dir, f)
    size_mb = os.path.getsize(full_path) / (1024*1024)
    print(f"  - {f} ({size_mb:.1f} MB)")

search_results 폴더 파일 수: 6개

n_insertion 관련 파일:
